# Exercise 5 - EKF Localization [9.0]

---

In [ ]:
from ex5 import *

#Autoloads evertyhing flaged with %aimport
%load_ext autoreload    
%autoreload 1               
%aimport ex5  

#Matplotlib output to cell
%matplotlib inline   

---
In this assignment you need to implement an EKF algorithm for localizing a robot in a given landmark map. 
The data for this exercise is recorded on a differential drive robot equipped with a sensor able to detect the distance and the angle of landmarks (e.g., beacons). The figure below visualizes the landmark map and the actual trajectory (ground truth) taken by the robot.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt


# read dataset
with open("dataset_2d_landmarks.p", "rb") as f:
    data = pickle.load(f)

# get landmark coordinates 
M = data['M']

# get ground truth trajectory
gt_traj = data['gt']

# show map
plt.figure(1)
plt.plot(M[:,0], M[:,1], 'black',marker="^",linestyle="")

# show ground truth trajectory
for i in range(0,len(gt_traj),10):
    plt.plot(gt_traj[i][0],gt_traj[i][1], '.r')


The following data is provided in `data`:

- *M* is the map of the environment where the robot must localize, it is a matrix of dimensions ($N_{landmarks}$, 2) where landmark of id =$id$ is $M[id] = [ x_{id} , y_{id} ]$ 
- *odom* is the noisy odometry readings observed by the robot during navigation in the form: $\hat{x}_t,\hat{y}_t, \hat{\theta}_t$ in the odometry frame
- *gt_traj* is the ground truth trajectory (poses in the map frame), you may want to use it for checking your results
- *z* are the sensor measurements

Each measurement $z_t$ contains a set of observed landmarks $[\rho_i; \phi_i; id_i]$, where $\rho_i$ is the measured distance, $\phi_i$ is the measured angle, and $id_i$ is the id of the landmark.

You can access the `data` as follows:

In [ ]:
# get odomety at timestamp 10
odom_10 = data['odom'][10]
print("Odom at step 10 is: \n",odom_10)

# get observation at timestamp 10
z_10 = data['z'][10]
print("Observation at step 10 is: \n",z_10)

---
## 5.1 Prediction [3.0]

The `ekf_predict` function computes a prediction about the robot's pose after moving by using the odometry motion model.

It takes as input:

- the current belief about the pose of the robot represented as a Gaussian distribution $\mathcal{N}(\mu_t,\Sigma_t)$ 
- the odometry readings $u_t$

The output is a prediction about the robot's pose $\mathcal{N}(\overline{\mu}_{t+1},\overline{\Sigma}_{t+1})$.

You can use your implementation of the `inverse_motion_model` function from Exercise 3 to compute the $u_t = [\delta_{rot1}, \delta_{trans}, \delta_{rot2}]$  from the odometry information.

Implement the `ekf_predict` function and verify that it is correct for the given test inputs. 

**Hint:** The uncertainty of the predicted pose be larger for the second time step!

In [ ]:
plot_result_ekf("predict")

---
## 5.2 Correction [3.0]

The `ekf_correct` implements the correction step of the EKF that corrects the prediction according to the sensor measurements.

It takes as input:

- the current prediction about the pose of the robot represented as a Gaussian distribution $\mathcal{N}(\overline{\mu}_{t+1},\overline{\Sigma}_{t+1})$
- the sensor measurements $z_t$

The output is new belief about the robot's pose $\mathcal{N}({\mu}_{t+1},{\Sigma}_{t+1})$.

Implement the `ekf_correct` function and normalize all angle quantities using `wrapToPi` to ensure rotation remain within the valid interval $[-\pi,\pi]$.

**Hint:** The correction step deacreases the uncertainty of the predicted pose.

In [ ]:
plot_result_ekf("predict&correct")

---
## 5.3 Localization [1.0]

Once you complete all the above functions, implement the main procedure of EKF localization `run_ekf_localization` which recursively estimates the pose of the robot using the odometry data and the sensor measurements. The following have to be defined in `init_params` as:

Assume the initial belief at time $t=0$ is:

- $\mu = [2, 2, \pi/2]'$
- $
\Sigma = \left(\begin{array}{cc} 
1 & 0 & 0\\
0 & 1 & 0 \\
0 & 0 & \pi/3
\end{array}\right)
$ 
            
The process noise $R$ and measurement noise $Q$ are defined as:
- $R = \left(\begin{array}{cc} 
\sigma_x^2 & 0 & 0 \\
0 & \sigma_y^2 & 0 \\
0 & 0 &  \sigma_{theta}^2
\end{array}\right)
$

with $\sigma_x = 0.25$ meters, $\sigma_y = 0.25$ meters and $\sigma_{theta} = 10$ degrees. 
 
- $Q = 
\left(\begin{array}{cc} 
\sigma_r^2 & 0 \\
0 & \sigma_{phi}^2 
\end{array}\right)
$

with $\sigma_r = 0.80$ meters, $\sigma_{phi} = 15$ degrees. 

In [ ]:
# Call EKF localization
run_ekf_localization()



---
## 5.4 Localization with Sensor limitations[2.0]
The previous system assumed a perfect sensor. In reality, the sensor may operate at a **different frequency** than the odometry and may occasionally **fail to provide measurements**.

The sensor specifications are defined as follows:
* **p_failure**: Probability (percentage) that a measurement is missing.
* **relative_frequency**: Sensor frequency relative to the odometry frequency.
* **relative_tolerance**: Time tolerance to accept sensor measurements that are slightly older than the current odometry timestep.

Update the function **`get_timesteps_system`** to include the following functionalities:
1. **Sensor Failure**: Simulate missing measurements according to **p_failure**.
2. **Frequency Matching**: Select sensor timesteps that are within the tolerance window of the odometry step. 

**Hint:** You can visually check if
    a) Failed sensor readings should not be matched 
    b) Matched sensor readings are within the tolerance (blue)

In [ ]:
print("50 % of sensor measurements fail.")
sensor_specs = {"p_failure":0.5,"relative_frequency":1,"relative_tolerance":0}
run_ekf_localization(sensor_specs=sensor_specs)

print("Sensor runs with a slighlty higher frequency")
sensor_specs = {"p_failure":0,"relative_frequency":1.33,"relative_tolerance":0.5}
run_ekf_localization(sensor_specs=sensor_specs)

print("Sensor runs with a slighlty higher frequency and 50 % of sensor measurements fail.")
sensor_specs = {"p_failure":0.5,"relative_frequency":1.33,"relative_tolerance":0.5}
run_ekf_localization(sensor_specs=sensor_specs)